# Plan 005 Interactive Trace Execution Host Contracts

This notebook is the pre-implementation contract artifact for Plan 005. It fixes the first-pass API shapes and stage artifact bindings before implementation starts.

## Final Runtime Payload

The host ultimately serves a `RunState` payload. The UI is a view over this object plus artifact payloads read by path; it is not the source of truth.

In [ ]:
run_state = {
    "ok": True,
    "run": {
        "run_id": "run_20260624_001",
        "status": "running",
        "current_stage": "extract",
        "output_dir": "output/workbench_runs/run_20260624_001",
    },
    "stages": [
        {"stage_id": "setup", "status": "complete", "output_refs": ["run_config.json"]},
        {"stage_id": "extract", "status": "ready", "output_refs": []},
        {"stage_id": "hypothesize", "status": "blocked", "requires": ["extract"]},
    ],
    "guides": {
        "extract": {
            "purpose": "Turn source text into typed causal objects.",
            "consumes": ["source_text", "source_packet_context"],
            "produces": ["ExtractionResult"],
            "audit_questions": ["Are evidence items traceable to source text?"],
        }
    },
}
run_state

## Create Run Request

A run starts with paths and options. Paths remain repo-local or absolute local paths resolved by the host. The source packet pins the research question when supplied.

In [ ]:
create_run_request = {
    "input_path": "input_text/source_packets/18_brumaire_source_packet.txt",
    "source_packet_path": "docs/source_packets/18_BRUMAIRE_SOURCE_PACKET.json",
    "research_question": "Why did the French Revolution culminate in Napoleon Bonaparte's 18 Brumaire coup?",
    "theories_path": "input_text/theories/18_brumaire_rival_frameworks.txt",
    "model": None,
    "refine": True,
    "max_budget": 1.0,
}
create_run_request

## Run Stage Response

Each stage writes a typed artifact and returns only compact summaries plus paths. Full structured artifacts stay on disk.

In [ ]:
run_stage_response = {
    "ok": True,
    "stage": {
        "stage_id": "extract",
        "status": "complete",
        "started_at": "2026-06-24T12:00:00Z",
        "completed_at": "2026-06-24T12:02:30Z",
        "input_refs": ["source_text.txt", "source_packet.json"],
        "output_refs": ["extraction.json"],
    },
    "artifacts": [
        {
            "artifact_id": "artifact_extract_001",
            "stage_id": "extract",
            "schema_name": "ExtractionResult",
            "path": "output/workbench_runs/run_20260624_001/extraction.json",
            "summary": "12 events, 50 evidence items, 4 mechanisms",
            "provenance": {"trace_id": "abc123", "model": "configured default"},
        }
    ],
}
run_stage_response

## Failure Shape

All endpoints return JSON failures with `ok: false`, `error_type`, and a human-readable `error`. LLM failures include the trace id when available.

In [ ]:
failure_response = {
    "ok": False,
    "error_type": "StageOrderError",
    "error": "stage 'test' requires completed hypothesis stage",
    "trace_id": None,
}
failure_response